In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("/Users/sanjanawadhawan/Downloads/France_Weather_Cleaned.csv")

In [3]:
df = pd.read_csv("/Users/sanjanawadhawan/Downloads/France_final_cleaned_data.csv")
df.head()

,Datetime,WMO_Station_ID,Temperature_C,Altitude_m,Latitude,Longitude,Hour,DayOfWeek,Month,DayOfYear,Year,IsWeekend
0,2018-01-01 00:00:00,61998,4.1,29,-49.352333,70.243333,0,0,1,1,2018,0
1,2018-01-01 00:00:00,7661,11.9,115,43.079333,5.940833,0,0,1,1,2018,0
2,2018-01-01 00:00:00,81408,25.4,6,3.890667,-51.804667,0,0,1,1,2018,0
3,2018-01-01 00:00:00,7149,8.4,89,48.716833,2.384333,0,0,1,1,2018,0
4,2018-01-01 00:00:00,7643,11.8,2,43.577000,3.963167,0,0,1,1,2018,0


In [4]:
df[["Latitude","Longitude"]].describe()

,Latitude,Longitude
count,742684.000000,742684.000000
mean,30.967395,1.341465
std,28.356788,34.675866
min,-66.663167,-62.852167
25%,16.335000,-1.608833
50%,44.581167,2.359833
75%,47.614333,5.959833
max,50.570000,140.001000


In [5]:
lat_min, lat_max = 41, 51
lon_min, lon_max = -5, 9

df_france = df[
    (df["Latitude"] >= lat_min) &
    (df["Latitude"] <= lat_max) &
    (df["Longitude"] >= lon_min) &
    (df["Longitude"] <= lon_max)
].copy()

In [6]:
print("Original rows:", len(df))
print("Filtered rows:", len(df_france))
print("Removed rows:", len(df) - len(df_france))

Original rows: 742684
Filtered rows: 514893
Removed rows: 227791


In [7]:
df_france[["Latitude","Longitude"]].describe()

,Latitude,Longitude
count,514893.000000,514893.000000
mean,46.382971,2.216889
std,2.366242,3.224037
min,41.918000,-4.412000
25%,44.118500,0.000000
50%,46.593833,2.359833
75%,48.445500,4.733000
max,50.570000,8.792667


In [8]:
df_france.to_csv("France_Weather_Metropolitan.csv", index=False)

In [9]:
def assign_region(lat, lon):

    # Latitude bands
    if lat < 44.8:
        lat_band = "S"
    elif lat < 47.7:
        lat_band = "C"
    else:
        lat_band = "N"

    # Longitude bands
    if lon < 0:
        lon_band = "W"
    elif lon < 4.4:
        lon_band = "C"
    else:
        lon_band = "E"

    return lat_band + lon_band

In [10]:
df_france["Region"] = df_france.apply(
    lambda row: assign_region(row["Latitude"], row["Longitude"]),
    axis=1
)

In [11]:
df_france["Region"].head()

1    SE
3    NC
4    SC
5    NE
6    SE
Name: Region, dtype: object

In [12]:
df_france["Region"].value_counts()

NC    89525
SC    89097
CC    76352
SE    70029
NW    62981
CW    50115
CE    38440
NE    25592
SW    12762
Name: Region, dtype: int64

In [13]:
df_france.to_csv("France_Weather_Metropolitan_Regions.csv", index=False)

In [14]:
weather_agg = df_france.groupby(['Datetime', 'Region'])['Temperature_C'].mean().reset_index()


In [15]:
weather_agg = df_france.groupby(['Datetime', 'Region'])['Temperature_C'].mean().reset_index()

In [16]:
weather_pivot = weather_agg.pivot(index='Datetime', columns='Region', values='Temperature_C')

In [17]:
df = pd.read_csv("France_Weather_Metropolitan_Regions.csv")
df.head()

,Datetime,WMO_Station_ID,Temperature_C,Altitude_m,Latitude,Longitude,Hour,DayOfWeek,Month,DayOfYear,Year,IsWeekend,Region
0,2018-01-01 00:00:00,7661,11.9,115,43.079333,5.940833,0,0,1,1,2018,0,SE
1,2018-01-01 00:00:00,7149,8.4,89,48.716833,2.384333,0,0,1,1,2018,0,NC
2,2018-01-01 00:00:00,7643,11.8,2,43.577000,3.963167,0,0,1,1,2018,0,SC
3,2018-01-01 00:00:00,7190,9.9,150,48.549500,7.640333,0,0,1,1,2018,0,NE
4,2018-01-01 00:00:00,7577,8.9,73,44.581167,4.733000,0,0,1,1,2018,0,SE


In [18]:
weather_agg = df.groupby(['Datetime', 'Region'])['Temperature_C'].mean().reset_index()
weather_agg.head()

,Datetime,Region,Temperature_C
0,2018-01-01 00:00:00,CC,7.083333
1,2018-01-01 00:00:00,CE,8.400000
2,2018-01-01 00:00:00,CW,10.150000
3,2018-01-01 00:00:00,NC,7.871429
4,2018-01-01 00:00:00,NE,9.200000


In [19]:
weather_pivot = weather_agg.pivot(index='Datetime', columns='Region', values='Temperature_C')
weather_pivot.head()

Region,CC,CE,CW,NC,NE,NW,SC,SE,SW
Datetime,,,,,,,,,
2018-01-01 00:00:00,7.083333,8.400000,10.15,7.871429,9.20,8.64,8.400000,8.300000,8.9
2018-01-01 03:00:00,6.366667,7.166667,10.10,7.142857,8.15,7.94,7.471429,8.383333,8.7
2018-01-01 06:00:00,5.916667,6.400000,11.00,6.300000,6.85,8.40,7.157143,7.966667,8.0
2018-01-01 09:00:00,6.966667,7.366667,10.90,6.742857,7.25,8.92,8.642857,8.566667,9.9
2018-01-01 12:00:00,8.200000,8.800000,10.25,7.314286,8.25,9.36,12.142857,11.616667,12.9


In [20]:
weather_pivot.columns = ['Temp_' + col for col in weather_pivot.columns]
weather_pivot.head()

,Temp_CC,Temp_CE,Temp_CW,Temp_NC,Temp_NE,Temp_NW,Temp_SC,Temp_SE,Temp_SW
Datetime,,,,,,,,,
2018-01-01 00:00:00,7.083333,8.400000,10.15,7.871429,9.20,8.64,8.400000,8.300000,8.9
2018-01-01 03:00:00,6.366667,7.166667,10.10,7.142857,8.15,7.94,7.471429,8.383333,8.7
2018-01-01 06:00:00,5.916667,6.400000,11.00,6.300000,6.85,8.40,7.157143,7.966667,8.0
2018-01-01 09:00:00,6.966667,7.366667,10.90,6.742857,7.25,8.92,8.642857,8.566667,9.9
2018-01-01 12:00:00,8.200000,8.800000,10.25,7.314286,8.25,9.36,12.142857,11.616667,12.9


In [21]:
weather_pivot = weather_pivot.reset_index()
weather_pivot.head()

,Datetime,Temp_CC,Temp_CE,Temp_CW,Temp_NC,Temp_NE,Temp_NW,Temp_SC,Temp_SE,Temp_SW
0,2018-01-01 00:00:00,7.083333,8.400000,10.15,7.871429,9.20,8.64,8.400000,8.300000,8.9
1,2018-01-01 03:00:00,6.366667,7.166667,10.10,7.142857,8.15,7.94,7.471429,8.383333,8.7
2,2018-01-01 06:00:00,5.916667,6.400000,11.00,6.300000,6.85,8.40,7.157143,7.966667,8.0
3,2018-01-01 09:00:00,6.966667,7.366667,10.90,6.742857,7.25,8.92,8.642857,8.566667,9.9
4,2018-01-01 12:00:00,8.200000,8.800000,10.25,7.314286,8.25,9.36,12.142857,11.616667,12.9


In [22]:
weather_pivot.to_csv("Weather_By_Region.csv", index=False)

In [23]:
import pandas as pd

energy_df = pd.read_csv("France_Electricity_Cleaned.csv")
energy_df.head()

,Datetime,Consumption_MW,Forecast_DayMinus1_MW,Forecast_Day_MW,Hour,Minute,DayOfWeek,Month,DayOfYear,IsWeekend,Lag_1,Lag_4,Lag_96,Lag_672
0,2018-01-15 00:00:00,66855.0,65200.0,65800.0,0,0,0,1,15,0,66943.0,64762.0,69455.0,61127.0
1,2018-01-15 00:30:00,65511.0,63600.0,64300.0,0,30,0,1,15,0,66855.0,65006.0,67432.0,59962.0
2,2018-01-15 01:00:00,63056.0,61500.0,62000.0,1,0,0,1,15,0,65511.0,67642.0,64505.0,57879.0
3,2018-01-15 01:30:00,62825.0,62400.0,63000.0,1,30,0,1,15,0,63056.0,66943.0,63880.0,57901.0
4,2018-01-15 02:00:00,62196.0,62300.0,62300.0,2,0,0,1,15,0,62825.0,66855.0,63114.0,57261.0


In [24]:
import pandas as pd

energy_df = pd.read_csv("France_Electricity_Cleaned.csv")
energy_df.head()

,Datetime,Consumption_MW,Forecast_DayMinus1_MW,Forecast_Day_MW,Hour,Minute,DayOfWeek,Month,DayOfYear,IsWeekend,Lag_1,Lag_4,Lag_96,Lag_672
0,2018-01-15 00:00:00,66855.0,65200.0,65800.0,0,0,0,1,15,0,66943.0,64762.0,69455.0,61127.0
1,2018-01-15 00:30:00,65511.0,63600.0,64300.0,0,30,0,1,15,0,66855.0,65006.0,67432.0,59962.0
2,2018-01-15 01:00:00,63056.0,61500.0,62000.0,1,0,0,1,15,0,65511.0,67642.0,64505.0,57879.0
3,2018-01-15 01:30:00,62825.0,62400.0,63000.0,1,30,0,1,15,0,63056.0,66943.0,63880.0,57901.0
4,2018-01-15 02:00:00,62196.0,62300.0,62300.0,2,0,0,1,15,0,62825.0,66855.0,63114.0,57261.0


In [25]:
weather_pivot['Datetime'] = pd.to_datetime(weather_pivot['Datetime'])
energy_df['Datetime'] = pd.to_datetime(energy_df['Datetime'])

In [26]:
weather_pivot['Datetime'] = pd.to_datetime(weather_pivot['Datetime'])
energy_df['Datetime'] = pd.to_datetime(energy_df['Datetime'])

In [27]:
weather_pivot = weather_pivot.set_index('Datetime')
energy_df = energy_df.set_index('Datetime')

In [28]:
weather_30min = weather_pivot.resample('30min').interpolate(method = 'linear')
weather_30min.head(10)

,Temp_CC,Temp_CE,Temp_CW,Temp_NC,Temp_NE,Temp_NW,Temp_SC,Temp_SE,Temp_SW
Datetime,,,,,,,,,
2018-01-01 00:00:00,7.083333,8.400000,10.150000,7.871429,9.200000,8.640000,8.400000,8.300000,8.900000
2018-01-01 00:30:00,6.963889,8.194444,10.141667,7.750000,9.025000,8.523333,8.245238,8.313889,8.866667
2018-01-01 01:00:00,6.844444,7.988889,10.133333,7.628571,8.850000,8.406667,8.090476,8.327778,8.833333
2018-01-01 01:30:00,6.725000,7.783333,10.125000,7.507143,8.675000,8.290000,7.935714,8.341667,8.800000
2018-01-01 02:00:00,6.605556,7.577778,10.116667,7.385714,8.500000,8.173333,7.780952,8.355556,8.766667
2018-01-01 02:30:00,6.486111,7.372222,10.108333,7.264286,8.325000,8.056667,7.626190,8.369444,8.733333
2018-01-01 03:00:00,6.366667,7.166667,10.100000,7.142857,8.150000,7.940000,7.471429,8.383333,8.700000
2018-01-01 03:30:00,6.291667,7.038889,10.250000,7.002381,7.933333,8.016667,7.419048,8.313889,8.583333
2018-01-01 04:00:00,6.216667,6.911111,10.400000,6.861905,7.716667,8.093333,7.366667,8.244444,8.466667


In [29]:
print("Weather range:", weather_30min.index.min(), "to", weather_30min.index.max())
print("Energy range:", energy_df.index.min(), "to", energy_df.index.max())

Weather range: 2018-01-01 00:00:00 to 2022-12-31 21:00:00
Energy range: 2018-01-15 00:00:00 to 2022-12-31 23:30:00


In [30]:
merged_df = energy_df.join(weather_30min, how='inner')
merged_df.head()

,Consumption_MW,Forecast_DayMinus1_MW,Forecast_Day_MW,Hour,Minute,DayOfWeek,Month,DayOfYear,IsWeekend,Lag_1,...,Lag_672,Temp_CC,Temp_CE,Temp_CW,Temp_NC,Temp_NE,Temp_NW,Temp_SC,Temp_SE,Temp_SW
Datetime,,,,,,,,,,,,,,,,,,,,,
2018-01-15 00:00:00,66855.0,65200.0,65800.0,0,0,0,1,15,0,66943.0,...,61127.0,4.266667,1.233333,6.925000,2.385714,0.250000,6.520000,5.771429,6.366667,6.200000
2018-01-15 00:30:00,65511.0,63600.0,64300.0,0,30,0,1,15,0,66855.0,...,59962.0,4.191667,1.094444,7.020833,2.530952,0.491667,6.663333,5.602381,6.275000,6.216667
2018-01-15 01:00:00,63056.0,61500.0,62000.0,1,0,0,1,15,0,65511.0,...,57879.0,4.116667,0.955556,7.116667,2.676190,0.733333,6.806667,5.433333,6.183333,6.233333
2018-01-15 01:30:00,62825.0,62400.0,63000.0,1,30,0,1,15,0,63056.0,...,57901.0,4.041667,0.816667,7.212500,2.821429,0.975000,6.950000,5.264286,6.091667,6.250000
2018-01-15 02:00:00,62196.0,62300.0,62300.0,2,0,0,1,15,0,62825.0,...,57261.0,3.966667,0.677778,7.308333,2.966667,1.216667,7.093333,5.095238,6.000000,6.266667


In [31]:
merged_df.isna().sum()

Consumption_MW           0
Forecast_DayMinus1_MW    0
Forecast_Day_MW          0
Hour                     0
Minute                   0
DayOfWeek                0
Month                    0
DayOfYear                0
IsWeekend                0
Lag_1                    0
Lag_4                    0
Lag_96                   0
Lag_672                  0
Temp_CC                  0
Temp_CE                  0
Temp_CW                  0
Temp_NC                  0
Temp_NE                  0
Temp_NW                  0
Temp_SC                  0
Temp_SE                  0
Temp_SW                  0
dtype: int64

In [32]:
merged_df = merged_df.reset_index()
merged_df.head()

,Datetime,Consumption_MW,Forecast_DayMinus1_MW,Forecast_Day_MW,Hour,Minute,DayOfWeek,Month,DayOfYear,IsWeekend,...,Lag_672,Temp_CC,Temp_CE,Temp_CW,Temp_NC,Temp_NE,Temp_NW,Temp_SC,Temp_SE,Temp_SW
0,2018-01-15 00:00:00,66855.0,65200.0,65800.0,0,0,0,1,15,0,...,61127.0,4.266667,1.233333,6.925000,2.385714,0.250000,6.520000,5.771429,6.366667,6.200000
1,2018-01-15 00:30:00,65511.0,63600.0,64300.0,0,30,0,1,15,0,...,59962.0,4.191667,1.094444,7.020833,2.530952,0.491667,6.663333,5.602381,6.275000,6.216667
2,2018-01-15 01:00:00,63056.0,61500.0,62000.0,1,0,0,1,15,0,...,57879.0,4.116667,0.955556,7.116667,2.676190,0.733333,6.806667,5.433333,6.183333,6.233333
3,2018-01-15 01:30:00,62825.0,62400.0,63000.0,1,30,0,1,15,0,...,57901.0,4.041667,0.816667,7.212500,2.821429,0.975000,6.950000,5.264286,6.091667,6.250000
4,2018-01-15 02:00:00,62196.0,62300.0,62300.0,2,0,0,1,15,0,...,57261.0,3.966667,0.677778,7.308333,2.966667,1.216667,7.093333,5.095238,6.000000,6.266667


In [33]:
merged_df = merged_df.ffill()

In [34]:
merged_df.isna().sum()

Datetime                 0
Consumption_MW           0
Forecast_DayMinus1_MW    0
Forecast_Day_MW          0
Hour                     0
Minute                   0
DayOfWeek                0
Month                    0
DayOfYear                0
IsWeekend                0
Lag_1                    0
Lag_4                    0
Lag_96                   0
Lag_672                  0
Temp_CC                  0
Temp_CE                  0
Temp_CW                  0
Temp_NC                  0
Temp_NE                  0
Temp_NW                  0
Temp_SC                  0
Temp_SE                  0
Temp_SW                  0
dtype: int64

In [35]:
merged_df.head()

,Datetime,Consumption_MW,Forecast_DayMinus1_MW,Forecast_Day_MW,Hour,Minute,DayOfWeek,Month,DayOfYear,IsWeekend,...,Lag_672,Temp_CC,Temp_CE,Temp_CW,Temp_NC,Temp_NE,Temp_NW,Temp_SC,Temp_SE,Temp_SW
0,2018-01-15 00:00:00,66855.0,65200.0,65800.0,0,0,0,1,15,0,...,61127.0,4.266667,1.233333,6.925000,2.385714,0.250000,6.520000,5.771429,6.366667,6.200000
1,2018-01-15 00:30:00,65511.0,63600.0,64300.0,0,30,0,1,15,0,...,59962.0,4.191667,1.094444,7.020833,2.530952,0.491667,6.663333,5.602381,6.275000,6.216667
2,2018-01-15 01:00:00,63056.0,61500.0,62000.0,1,0,0,1,15,0,...,57879.0,4.116667,0.955556,7.116667,2.676190,0.733333,6.806667,5.433333,6.183333,6.233333
3,2018-01-15 01:30:00,62825.0,62400.0,63000.0,1,30,0,1,15,0,...,57901.0,4.041667,0.816667,7.212500,2.821429,0.975000,6.950000,5.264286,6.091667,6.250000
4,2018-01-15 02:00:00,62196.0,62300.0,62300.0,2,0,0,1,15,0,...,57261.0,3.966667,0.677778,7.308333,2.966667,1.216667,7.093333,5.095238,6.000000,6.266667


In [36]:
merged_df.to_csv("Merged_Energy_Weather_30min.csv", index=False)

In [37]:
import pandas as pd
import numpy as np

# Make sure Datetime is datetime type
merged_df['Datetime'] = pd.to_datetime(merged_df['Datetime'])

# Sort by time
merged_df = merged_df.sort_values('Datetime').reset_index(drop=True)

# Quick check
print(merged_df[['Datetime']].head())
print(merged_df[['Datetime']].tail())


             Datetime
0 2018-01-15 00:00:00
1 2018-01-15 00:30:00
2 2018-01-15 01:00:00
3 2018-01-15 01:30:00
4 2018-01-15 02:00:00
                 Datetime
84278 2022-12-31 19:00:00
84279 2022-12-31 19:30:00
84280 2022-12-31 20:00:00
84281 2022-12-31 20:30:00
84282 2022-12-31 21:00:00


In [38]:
feature_cols = [
    'Consumption_MW',
    'Temp_CC', 'Temp_CE', 'Temp_CW',
    'Temp_NC', 'Temp_NE', 'Temp_NW',
    'Temp_SC', 'Temp_SE', 'Temp_SW',
    'Hour', 'DayOfWeek', 'Month', 'IsWeekend'
]

target_col = 'Consumption_MW'

In [39]:
train_df = merged_df[merged_df['Datetime'] < '2022-01-01'].copy()
val_df   = merged_df[(merged_df['Datetime'] >= '2022-01-01') & (merged_df['Datetime'] < '2022-07-01')].copy()
test_df  = merged_df[merged_df['Datetime'] >= '2022-07-01'].copy()

print("Train shape:", train_df.shape)
print("Val shape:  ", val_df.shape)
print("Test shape: ", test_df.shape)

print("\nTrain date range:", train_df['Datetime'].min(), "to", train_df['Datetime'].max())
print("Val date range:  ", val_df['Datetime'].min(), "to", val_df['Datetime'].max())
print("Test date range: ", test_df['Datetime'].min(), "to", test_df['Datetime'].max())

Train shape: (67440, 23)
Val shape:   (8016, 23)
Test shape:  (8827, 23)

Train date range: 2018-01-15 00:00:00 to 2021-12-31 23:30:00
Val date range:   2022-01-15 00:00:00 to 2022-06-30 23:30:00
Test date range:  2022-07-01 00:00:00 to 2022-12-31 21:00:00


In [40]:
LOOKBACK = 48   # past 24 hours
HORIZON = 48    # predict next 24 hours


In [41]:
def create_sequences(df, feature_cols, target_col, lookback=48, horizon=48):
    X, y = [], []
    
    feature_array = df[feature_cols].values
    target_array = df[target_col].values
    datetime_array = df['Datetime'].values
    
    for i in range(len(df) - lookback - horizon + 1):
        X_seq = feature_array[i : i + lookback]
        y_seq = target_array[i + lookback : i + lookback + horizon]
        
        X.append(X_seq)
        y.append(y_seq)
    
    X = np.array(X)
    y = np.array(y)
    
    return X, y

In [42]:
X_train, y_train = create_sequences(train_df, feature_cols, target_col, LOOKBACK, HORIZON)
X_val, y_val     = create_sequences(val_df, feature_cols, target_col, LOOKBACK, HORIZON)
X_test, y_test   = create_sequences(test_df, feature_cols, target_col, LOOKBACK, HORIZON)

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

print("X_val shape:", X_val.shape)
print("y_val shape:", y_val.shape)

print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

X_train shape: (67345, 48, 14)
y_train shape: (67345, 48)
X_val shape: (7921, 48, 14)
y_val shape: (7921, 48)
X_test shape: (8732, 48, 14)
y_test shape: (8732, 48)


In [43]:
from sklearn.preprocessing import StandardScaler
import numpy as np

# -----------------------------
# 1. Initialize scaler
# -----------------------------
x_scaler = StandardScaler()

# -----------------------------
# 2. Flatten X_train so scaler can fit feature-wise
#    Original shape: (samples, 48, 14)
#    Flattened:      (samples*48, 14)
# -----------------------------
X_train_2d = X_train.reshape(-1, X_train.shape[-1])
X_val_2d   = X_val.reshape(-1, X_val.shape[-1])
X_test_2d  = X_test.reshape(-1, X_test.shape[-1])

# -----------------------------
# 3. Fit ONLY on training data
# -----------------------------
x_scaler.fit(X_train_2d)

# -----------------------------
# 4. Transform train / val / test
# -----------------------------
X_train_scaled = x_scaler.transform(X_train_2d).reshape(X_train.shape)
X_val_scaled   = x_scaler.transform(X_val_2d).reshape(X_val.shape)
X_test_scaled  = x_scaler.transform(X_test_2d).reshape(X_test.shape)

# -----------------------------
# 5. Check shapes
# -----------------------------
print("X_train_scaled shape:", X_train_scaled.shape)
print("X_val_scaled shape:  ", X_val_scaled.shape)
print("X_test_scaled shape: ", X_test_scaled.shape)

X_train_scaled shape: (67345, 48, 14)
X_val_scaled shape:   (7921, 48, 14)
X_test_scaled shape:  (8732, 48, 14)


In [44]:
print("Before scaling:")
print(X_train[0, 0, :5])

print("\nAfter scaling:")
print(X_train_scaled[0, 0, :5])

Before scaling:
[6.68550000e+04 4.26666667e+00 1.23333333e+00 6.92500000e+00
 2.38571429e+00]

After scaling:
[ 1.25752285 -0.85604026 -1.15476763 -0.95956159 -1.14956   ]


In [45]:
print("Mean:", X_train_scaled.mean())
print("Std:", X_train_scaled.std())

Mean: 1.3616659524622297e-14
Std: 0.9999999999998603


In [46]:
print("Raw X_train shape:", X_train.shape)
print("Scaled X_train shape:", X_train_scaled.shape)

print("Overall mean of X_train_scaled:", X_train_scaled.mean())
print("Overall std of X_train_scaled:", X_train_scaled.std())

print("Mean by feature:")
print(X_train_scaled.reshape(-1, X_train_scaled.shape[-1]).mean(axis=0))

print("Std by feature:")
print(X_train_scaled.reshape(-1, X_train_scaled.shape[-1]).std(axis=0))

Raw X_train shape: (67345, 48, 14)
Scaled X_train shape: (67345, 48, 14)
Overall mean of X_train_scaled: 1.3616659524622297e-14
Overall std of X_train_scaled: 0.9999999999998603
Mean by feature:
[ 1.69259308e-14 -3.62167502e-13  2.23584633e-13 -3.99558952e-14
 -9.44370423e-13 -2.74879945e-13  8.58918069e-14  1.13098520e-12
 -2.39724623e-13  8.43618781e-13 -7.70990501e-18  1.11870126e-14
 -1.53076055e-12 -4.97329397e-14]
Std by feature:
[1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


In [47]:
print(X_train_nbeats.shape)
print(y_train.shape)

NameError: name 'X_train_nbeats' is not defined

In [ ]:
X_train_nbeats = X_train_scaled.reshape(X_train_scaled.shape[0], -1)

In [ ]:
print(X_train_nbeats.shape)
print(y_train.shape)

In [ ]:
np.save("X_train_scaled.npy", X_train_scaled)

In [ ]:
import os
print(os.getcwd())
print(os.listdir())

In [48]:
merged_df.to_csv("/Users/sanjanawadhawan/Downloads/Merged_Energy_Weather_30min_INTERPOLATED.csv", index=False)